# Lab 6 — 나만의 샘플러 만들기

**확률통계 · Topic 6 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. `rng.random()` (균등난수) **하나만으로** 지수분포를 만든다.
2. **Cauchy** 샘플러를 만들고, 표본평균이 **수렴하지 않는 것**을 목격한다.
3. **Memoryless property** 를 시뮬레이션으로 확인한다.

⏱ **예상 소요 시간: 35분**

> 오늘 쓰는 도구는 딱 하나다 — **역함수를 취한 CDF**.
> $$X = F_X^{-1}(U), \qquad U \sim U(0,1)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(20260302)

u = rng.random(5)
print("균등난수 5개:", u)

## Part 1. 지수분포 샘플러

$F_X(x) = 1 - e^{-\lambda x}$ 를 뒤집으면

$$x = -\frac{\ln(1-u)}{\lambda}$$

### 실습 1 — 공식을 코드로

In [ ]:
LAM = 0.7

def sample_exponential(n, lam, rng):
    u = rng.random(n)
    # TODO 1: 역함수 공식 -ln(1-u)/lam 을 적용해 돌려주세요
    return np.zeros(n)


x = sample_exponential(200_000, LAM, rng)

print(f"표본평균   {x.mean():.4f}   (이론 1/lambda = {1 / LAM:.4f})")
print(f"표본표준편차 {x.std():.4f}   (이론 1/lambda = {1 / LAM:.4f})")

### 실습 2 — 이론 PDF와 겹쳐 보기

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(x, bins=np.linspace(0, 8, 80), density=True, alpha=0.8, label="my sampler")
xs = np.linspace(0, 8, 300)
# TODO 2: 이론 PDF lam*exp(-lam*x) 를 겹쳐 그리세요
#         힌트: plt.plot(xs, LAM * np.exp(-LAM * xs), label="theory PDF")

plt.xlabel("x")
plt.ylabel("density")
plt.title("Exponential from rng.random() only")
plt.legend()
plt.show()

> SciPy와도 비교해보자. `stats.expon(scale=1/LAM).rvs(...)` 와 모양이 같으면 성공이다.

## Part 2. Memoryless property

**"이미 $s$ 만큼 기다렸다"는 사실이 앞으로의 대기시간을 바꾸지 않는다.**

$$\mathbb{P}[X > s + t \mid X > s] = \mathbb{P}[X > t]$$

시뮬레이션으로 확인하자. 조건부 확률은 **"조건을 만족하는 것만 골라내서 비율을 세는 것"** 이었다.

### 실습 3

In [ ]:
s, t = 2.0, 1.0

# 조건: X > s 인 표본만 골라낸다
survived = x[x > s]

# TODO 3: 그 표본들 중에서 s+t 보다 큰 것의 비율을 구하세요
#         힌트: (survived > s + t).mean()
cond_prob = 0.0

print(f"P[X > {s + t} | X > {s}] = {cond_prob:.4f}   (조건을 만족한 표본 {len(survived)}개)")
print(f"P[X > {t}]              = {(x > t).mean():.4f}")
print(f"이론값 exp(-lam*t)      = {np.exp(-LAM * t):.4f}")

🤔 **두 값이 거의 같다.** 2시간을 기다렸든 방금 왔든, 앞으로 1시간 더 기다릴 확률은 같다.

> 버스를 오래 기다렸다고 "이제 곧 오겠지"라고 생각하는 것은
> 배차가 memoryless라면 **근거 없는 기대**다.

## Part 3. Cauchy — 평균이 없는 분포

Cauchy 분포의 CDF를 뒤집으면 이렇게 된다.

$$x = \tan\left(\pi\left(u - \tfrac{1}{2}\right)\right)$$

모양은 종 모양이라 정규분포와 비슷해 보인다. 그런데 **꼬리가 훨씬 두껍다.**

### 실습 4 — Cauchy 샘플러

In [ ]:
def sample_cauchy(n, rng):
    u = rng.random(n)
    # TODO 4: tan(pi * (u - 0.5)) 를 돌려주세요
    return np.zeros(n)


c = sample_cauchy(200_000, rng)

print(f"중앙값(median) {np.median(c):.4f}")
print(f"표본평균        {c.mean():.4f}")
print(f"최댓값          {c.max():.1f}")
print(f"최솟값          {c.min():.1f}")

최댓값과 최솟값을 보자. **터무니없이 큰 값**이 섞여 있다.
이런 극단값 하나가 평균을 통째로 끌고 간다.

### 실습 5 — 표본평균은 수렴하는가

Topic 1부터 봐온 "누적 평균" 그림을 Cauchy로 그려보자.

In [ ]:
n = np.arange(1, len(c) + 1)

# TODO 5: Cauchy 표본의 누적 평균을 구하세요.  힌트: np.cumsum(c) / n
running_cauchy = np.zeros(len(c))

normal = rng.normal(0, 1, size=len(c))
running_normal = np.cumsum(normal) / n

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
axes[0].plot(n, running_normal)
axes[0].axhline(0, color="red", ls="--")
axes[0].set_xscale("log")
axes[0].set_title("Normal - converges")
axes[0].set_xlabel("n")
axes[0].set_ylabel("running mean")

axes[1].plot(n, running_cauchy, color="darkorange")
axes[1].axhline(0, color="red", ls="--")
axes[1].set_xscale("log")
axes[1].set_title("Cauchy - never settles")
axes[1].set_xlabel("n")
plt.show()

🤯 **왼쪽은 0으로 수렴하는데, 오른쪽은 20만 개를 뽑아도 안정되지 않는다.**

가끔 등장하는 거대한 값이 그때까지의 평균을 통째로 흔들어버리기 때문이다.
Cauchy는 **기댓값 자체가 존재하지 않는 분포**다 (적분이 발산한다).

> Topic 1부터 "많이 반복하면 평균이 안정된다"고 봐 왔다.
> **그것이 항상 참은 아니다.** 언제 참이고 언제 거짓인지 —
> **Topic 10 대수의 법칙과 중심극한정리**에서 정확한 조건을 배운다.
> 오늘 본 Cauchy가 그때 **반례**로 다시 등장한다.

---

## 마무리 — 자가 점검

- [ ] $X = F^{-1}(U)$ 로 원하는 분포를 만들 수 있다
- [ ] 내가 만든 표본이 이론 PDF와 겹치는 것을 확인했다
- [ ] memoryless property를 시뮬레이션으로 확인했다
- [ ] Cauchy에서는 표본평균이 수렴하지 않는 것을 보았다

**오늘 가장 놀라웠던 점을 한 문장으로.**

> (여기에 작성)